In [3]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from scipy.stats import poisson
from sklearn.neighbors import KernelDensity
import time
import dill
from scipy.optimize import minimize

import pickle

In [4]:
metadata = pd.read_csv('../../Allen_institute_labels_v2_classsubclasssupertype.csv')

In [6]:
#create a dictionary that outputs the levels of the allen institute heirachy going up to hypothalamic or non-hypothalamic
hypo = ['15 HY Gnrh1 Glut','16 HY MM Glut','14 HY Glut','13 CNU-HYa Glut','12 HY GABA','11 CNU-HYa GABA']
t0 = time.time()
a = 0
parent_dict = {}
for item in metadata.index:
    a += 1
    parent_dict[metadata.loc[item,'cluster_alias']]= metadata.loc[item,'supertype_id_label']
    parent_dict[metadata.loc[item,'supertype_id_label']] = metadata.loc[item,'subclass_id_label']
    parent_dict[metadata.loc[item,'subclass_id_label']] = metadata.loc[item,'class_id_label']
    if metadata.loc[item,'class_id_label'] in hypo:
        parent_dict[metadata.loc[item,'class_id_label']] = 'hypo'
    else:
        parent_dict[metadata.loc[item,'class_id_label']] = 'not hypo'
    if a % 100000 == 0:
        t1 = time.time()
        print(str(a/len(metadata)) + ' percent in ' + str(t1-t0) + ' seconds')
parent_dict['Unlabeled'] = 'Unlabeled'

0.024734255162533737 percent in 10.054056406021118 seconds
0.04946851032506747 percent in 20.138827800750732 seconds
0.07420276548760121 percent in 30.138044357299805 seconds
0.09893702065013495 percent in 40.087825298309326 seconds
0.1236712758126687 percent in 50.10247778892517 seconds
0.14840553097520243 percent in 60.23152303695679 seconds
0.17313978613773617 percent in 70.30083799362183 seconds
0.1978740413002699 percent in 80.52759289741516 seconds
0.22260829646280364 percent in 90.74825477600098 seconds
0.2473425516253374 percent in 100.98869776725769 seconds
0.2720768067878711 percent in 111.0748450756073 seconds
0.29681106195040485 percent in 121.22237253189087 seconds
0.3215453171129386 percent in 131.49449515342712 seconds
0.34627957227547235 percent in 141.75285863876343 seconds
0.37101382743800604 percent in 152.04313564300537 seconds
0.3957480826005398 percent in 162.00763845443726 seconds
0.42048233776307353 percent in 172.13280725479126 seconds
0.4452165929256073 percen

In [5]:
import pickle

with open('../../parent_dict_v2.pkl', 'wb') as f:
    pickle.dump(parent_dict, f)